# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [15]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.datasets import  fetch_california_housing
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

#ML Supervised lesson
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

#Ensemble Lesson
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

In [16]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [17]:
spaceship_clean = spaceship.dropna()
print (spaceship_clean.isna().sum())
spaceship_clean['Cabin'].unique()
spaceship_clean['Cabin'] = spaceship_clean['Cabin'].str[0]
spaceship_clean['Cabin'].unique()

spaceship_clean = spaceship.drop(columns=['PassengerId', 'Name'])

spaceship_clean_dummies = pd.get_dummies(spaceship_clean, drop_first=False)

target = 'Transported'

features = spaceship_clean_dummies.drop(columns=[target])

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64


/var/folders/6z/jl55nn756jggg02pwd35rwm40000gn/T/ipykernel_61772/1463342286.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spaceship_clean['Cabin'] = spaceship_clean['Cabin'].str[0]


**Perform Train Test Split**

In [18]:
from sklearn.model_selection import train_test_split

mask = spaceship_clean_dummies[target].notna()
x=features.loc[mask]
y=spaceship_clean_dummies.loc[mask,target]
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42, stratify=y
)   
print("NaNs in X:", x.isna().sum().sum(), " | NaNs in y:", y.isna().sum())



NaNs in X: 1122  | NaNs in y: 0


In [19]:


from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Pipeline: impute → scale → KNN
knn_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # fill NaNs
    ('scaler', StandardScaler()),                   # scale features
    ('knn', KNeighborsClassifier(n_neighbors=5))    # model
])

# Train
knn_pipe.fit(X_train, y_train)

# Predict
y_pred = knn_pipe.predict(X_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

normalizer = MinMaxScaler()
normalizer.fit(X_train)


X_train_norm = normalizer.transform(X_train)
X_test_norm = normalizer.transform(X_test)



Accuracy: 0.6877515813686026

Classification Report:
               precision    recall  f1-score   support

       False       0.66      0.75      0.70       863
        True       0.72      0.63      0.67       876

    accuracy                           0.69      1739
   macro avg       0.69      0.69      0.69      1739
weighted avg       0.69      0.69      0.69      1739



**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [35]:
bagging_reg = BaggingRegressor(DecisionTreeRegressor(max_depth=8), n_estimators=100, max_samples=3000, bootstrap=True)
bagging_reg.fit(X_train_norm, y_train)
pred = bagging_reg.predict  (X_test_norm)
print("TEST DATA")

print("R2 score ", r2_score(y_test, pred))
print("RMSE ", root_mean_squared_error(y_test, pred))
print("MAE ", mean_absolute_error(y_test, pred))


print("----------------------------")

print("TRAIN DATA")
pred = bagging_reg.predict(X_train_norm)

print("R2 score ", r2_score(y_train, pred))
print("RMSE ", root_mean_squared_error(y_train, pred))
print("MAE ", mean_absolute_error(y_train, pred))

TEST DATA
R2 score  0.43739461357272513
RMSE  0.37502464778456907
MAE  0.2782679745750292
----------------------------
TRAIN DATA
R2 score  0.4948691924055668
RMSE  0.3553535892349759
MAE  0.2663706166849555


- Random Forests

In [39]:
rforest = RandomForestRegressor(n_estimators=100, max_depth=8)
rforest.fit(X_train_norm, y_train)

print("TEST DATA")

print("R2 score ", r2_score(y_test, pred))
print("RMSE ", root_mean_squared_error(y_test, pred))
print("MAE ", mean_absolute_error(y_test, pred))


print("----------------------------")

print("TRAIN DATA")
pred = rforest.predict(X_train_norm)

print("R2 score ", r2_score(y_train, pred))
print("RMSE ", root_mean_squared_error(y_train, pred))
print("MAE ", mean_absolute_error(y_train, pred))

TEST DATA
R2 score  0.43527795722039486
RMSE  0.3757294518163473
MAE  0.2794522012102683
----------------------------
TRAIN DATA
R2 score  0.5023616122569636
RMSE  0.3527083288702133
MAE  0.2651781079759874


- Gradient Boosting

In [51]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

pipe = make_pipeline(
    SimpleImputer(strategy="median"),   # handles NaNs
    GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=0)
    )
pipe.fit(X_train_norm, y_train)
# Evaluate on test
pred_test = pipe.predict(X_test_norm)
print("TEST R2 :",  r2_score(y_test, pred_test))
print("TEST RMSE:", root_mean_squared_error(y_test, pred_test))
print("TEST MAE :", mean_absolute_error(y_test, pred_test))

# Evaluate on train
pred_train = pipe.predict(X_train_norm)
print("TRAIN R2 :",  r2_score(y_train, pred_train))
print("TRAIN RMSE:", root_mean_squared_error(y_train, pred_train))
print("TRAIN MAE :", mean_absolute_error(y_train, pred_train))

TEST R2 : 0.43311967375020655
TEST RMSE: 0.37644675808987604
TEST MAE : 0.2956615417225035
TRAIN R2 : 0.47448724608575954
TRAIN RMSE: 0.3624519230705649
TRAIN MAE : 0.28747383486931716


- Adaptive Boosting

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error


ada = make_pipeline(
    SimpleImputer(strategy="median"),   # handles NaNs
    AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=1, random_state=0),
        n_estimators=300,
        learning_rate=0.05,
        random_state=0
    )
)

ada.fit(X_train_norm, y_train)


pred_test = ada.predict(X_test_norm)    
print("TEST R2 :",  r2_score(y_test, pred_test))
print("TEST RMSE:", root_mean_squared_error(y_test, pred_test))
print("TEST MAE :", mean_absolute_error(y_test, pred_test))


pred_train = ada.predict(X_train_norm)  
print("TRAIN R2 :",  r2_score(y_train, pred_train))
print("TRAIN RMSE:", root_mean_squared_error(y_train, pred_train))
print("TRAIN MAE :", mean_absolute_error(y_train, pred_train))

TEST R2 : 0.2028359620535618
TEST RMSE: 0.4464077422115159
TEST MAE : 0.3967762541689138
TRAIN R2 : 0.2139117438676078
TRAIN RMSE: 0.4432966324123136
TRAIN MAE : 0.39365184284208155


Which model is the best and why?

In [ ]:
	# •	Best so far: Random Forest (R² ≈ 0.43 test).
